# Module 12 - 실습 2 솔루션: 서브그래프 패턴

In [ ]:
import sys
sys.path.insert(0, '../..')

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from common.fake_llm import FakeLLM

print("서브그래프 솔루션 준비 완료")

In [ ]:
class AnalysisSubState(TypedDict):
    """분석 서브그래프의 격리된 상태."""
    text: str
    keywords: list[str] | None
    summary: str | None
    sentiment: str | None

def extract_keywords_node(state: AnalysisSubState) -> dict:
    """키워드를 추출합니다."""
    text = state["text"]
    words = text.split()
    keywords = [w for w in words if len(w) > 3][:5]
    print(f"  [서브그래프] 키워드 추출: {keywords}")
    return {"keywords": keywords}

def summarize_node(state: AnalysisSubState) -> dict:
    """텍스트를 요약합니다."""
    text = state["text"]
    summary = text[:100] + "..." if len(text) > 100 else text
    print(f"  [서브그래프] 요약 생성 완료")
    return {"summary": summary}

def analyze_sentiment_node(state: AnalysisSubState) -> dict:
    """감성을 분석합니다."""
    text = state["text"].lower()
    if any(w in text for w in ["좋", "훌륭", "만족", "good", "great"]):
        sentiment = "positive"
    elif any(w in text for w in ["나쁜", "불만", "실망", "bad", "poor"]):
        sentiment = "negative"
    else:
        sentiment = "neutral"
    print(f"  [서브그래프] 감성 분석: {sentiment}")
    return {"sentiment": sentiment}

In [ ]:
def build_analysis_subgraph():
    """분석 서브그래프를 빌드합니다."""
    graph = StateGraph(AnalysisSubState)
    
    graph.add_node("extract_keywords", extract_keywords_node)
    graph.add_node("summarize", summarize_node)
    graph.add_node("analyze_sentiment", analyze_sentiment_node)
    
    graph.set_entry_point("extract_keywords")
    graph.add_edge("extract_keywords", "summarize")
    graph.add_edge("summarize", "analyze_sentiment")
    graph.add_edge("analyze_sentiment", END)
    
    return graph.compile()

sub_app = build_analysis_subgraph()
sub_result = sub_app.invoke({
    "text": "이 제품은 정말 훌륭합니다. 성능이 좋고 만족스럽습니다.",
    "keywords": None,
    "summary": None,
    "sentiment": None,
})
print(f"키워드: {sub_result.get('keywords')}")
print(f"요약: {sub_result.get('summary')}")
print(f"감성: {sub_result.get('sentiment')}")

In [ ]:
assert sub_result.get("keywords") is not None
assert sub_result.get("summary") is not None
assert sub_result.get("sentiment") in ["positive", "negative", "neutral"]
print("✅ 실습 2 완료! 서브그래프가 독립적으로 동작합니다.")

In [ ]:
class MainState(TypedDict):
    document_id: str
    raw_text: str
    cleaned_text: str | None
    text: str
    keywords: list[str] | None
    summary: str | None
    sentiment: str | None
    report: str | None

def preprocess_node(state: MainState) -> dict:
    """텍스트를 전처리합니다."""
    raw = state["raw_text"]
    cleaned = raw.strip().replace("\n", " ")
    print(f"[메인] 전처리 완료: {len(cleaned)}자")
    return {"cleaned_text": cleaned, "text": cleaned}

def generate_report_node(state: MainState) -> dict:
    """최종 리포트를 생성합니다."""
    report = (
        f"문서 분석 리포트 (ID: {state['document_id']})\n"
        f"{'='*50}\n"
        f"키워드: {', '.join(state.get('keywords') or [])}\n"
        f"요약: {state.get('summary', 'N/A')}\n"
        f"감성: {state.get('sentiment', 'N/A')}\n"
    )
    print(f"[메인] 리포트 생성 완료")
    return {"report": report}

def build_main_graph():
    """서브그래프가 포함된 메인 그래프를 구성합니다."""
    graph = StateGraph(MainState)
    
    analysis_subgraph = build_analysis_subgraph()
    graph.add_node("preprocess", preprocess_node)
    graph.add_node("analyze", analysis_subgraph)  # 서브그래프를 노드로 등록!
    graph.add_node("generate_report", generate_report_node)
    
    graph.set_entry_point("preprocess")
    graph.add_edge("preprocess", "analyze")
    graph.add_edge("analyze", "generate_report")
    graph.add_edge("generate_report", END)
    
    checkpointer = MemorySaver()
    return graph.compile(checkpointer=checkpointer)

app = build_main_graph()
print("메인 그래프 구성 완료")

In [ ]:
result = app.invoke(
    {
        "document_id": "DOC-001",
        "raw_text": "이 제품은 정말 훌륭합니다. 성능이 좋고 디자인이 만족스럽습니다.",
        "cleaned_text": None,
        "text": "",
        "keywords": None,
        "summary": None,
        "sentiment": None,
        "report": None,
    },
    config={"configurable": {"thread_id": "doc-001"}},
)

print(result.get("report", "리포트가 없습니다"))

In [ ]:
assert result.get("report") is not None
assert result.get("keywords") is not None
assert result.get("sentiment") is not None
print(f"✅ 실습 3 완료! 서브그래프가 메인 그래프에 통합되었습니다. (감성: {result['sentiment']})")

## 보너스: 동적 그래프 구성 (Configurable)

실행 시점에 config 값에 따라 노드를 추가하거나 제거하는 동적 그래프를 구성할 수 있습니다.

예를 들어, `enable_validation` 옵션이 `True`일 때만 검증 노드를 포함시키는 식으로
그래프를 런타임에 유연하게 빌드할 수 있습니다.

In [ ]:
# 솔루션: config에 따라 노드를 추가/제거하는 동적 그래프

from typing import TypedDict as _TypedDict
from langgraph.graph import StateGraph as _StateGraph, END as _END


class DynamicState(_TypedDict):
    value: str
    validated: bool


def process_node(state: DynamicState) -> dict:
    """기본 처리 노드."""
    return {"value": f"처리됨: {state['value']}"}


def validate_fn(state: DynamicState) -> dict:
    """선택적 검증 노드."""
    print("  [validate] 검증 수행 중...")
    return {"validated": True}


def build_dynamic_graph(config: dict):
    """config에 따라 동적으로 그래프를 구성합니다."""
    enable_validation = config.get("enable_validation", False)

    graph = _StateGraph(DynamicState)
    graph.add_node("process", process_node)

    if enable_validation:
        # 검증 노드를 동적으로 추가
        graph.add_node("validate", validate_fn)
        graph.set_entry_point("process")
        graph.add_edge("process", "validate")
        graph.add_edge("validate", _END)
    else:
        graph.set_entry_point("process")
        graph.add_edge("process", _END)

    return graph.compile()


# 검증 노드 없이 실행
graph_without_validation = build_dynamic_graph({"enable_validation": False})

# 검증 노드 포함하여 실행
graph_with_validation = build_dynamic_graph({"enable_validation": True})

result_no_val = graph_without_validation.invoke({"value": "테스트 입력", "validated": False})
result_with_val = graph_with_validation.invoke({"value": "테스트 입력", "validated": False})

print(f"검증 없음: {result_no_val}")
print(f"검증 포함: {result_with_val}")

In [ ]:
# 검증 셀
assert graph_without_validation is not None, "build_dynamic_graph({'enable_validation': False})를 구현하세요"
assert graph_with_validation is not None, "build_dynamic_graph({'enable_validation': True})를 구현하세요"

# 두 그래프가 서로 다른 노드 구성을 가져야 함
nodes_without = set(graph_without_validation.get_graph().nodes.keys())
nodes_with = set(graph_with_validation.get_graph().nodes.keys())
assert "validate" not in nodes_without, "'validate' 노드는 enable_validation=False 일 때 없어야 합니다"
assert "validate" in nodes_with, "'validate' 노드는 enable_validation=True 일 때 있어야 합니다"

print("✅ 보너스 완료! 동적 그래프 구성이 올바르게 동작합니다.")